# Laboratorio 3: atención en BERT

- Nelson García 22434
- Joaquín Puente 22296


## Parte A — Tokenización



### 1. Selección reproducible del corpus




In [1]:
from pathlib import Path
import random
import re
import sys

import pandas as pd
import transformers
from IPython.display import display
from transformers import AutoTokenizer

SEMILLA = 20260831
TOTAL_PAGINAS = 43
rng = random.Random(SEMILLA)
pagina_inicial = rng.randint(2, TOTAL_PAGINAS - 1)
paginas = (pagina_inicial, pagina_inicial + 1)

print(f"Python: {sys.version.split()[0]}")
print(f"Transformers: {transformers.__version__}")
print(f"Páginas seleccionadas: {paginas[0]} y {paginas[1]}")

/home/nelson/Documents/Uvg/Procesamiento_Lenguaje/Lab_3_NLP/.venv-part-a/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.13
Transformers: 5.16.1
Páginas seleccionadas: 14 y 15


In [2]:
ruta_corpus = Path("corpus_paginas_14_15.txt")
texto_paginas = ruta_corpus.read_text(encoding="utf-8")

print(f"Archivo: {ruta_corpus}")
print(f"Caracteres: {len(texto_paginas):,}")
print("\nVista previa del corpus:\n")
print(texto_paginas[:900])

Archivo: corpus_paginas_14_15.txt
Caracteres: 5,925

Vista previa del corpus:

cuenta de que a su lado el café salía de la cafetera volcada, derramándose
sobre la alfombra.
      —¡Madre! ¡Madre! —gimió Gregorio, mirándola desde abajo. Por un
momento se olvidó del gerente; y no pudo evita, ante el café vertido, abrir y
cerrar repetidas veces las mandíbulas en el vacío. Su madre, gritando de
nuevo y huyendo de la mesa, se lanzó en brazos del padre, que corrió a su
encuentro. Pero Gregorio no podía dedicar ya su atención a sus padres; el
gerente estaba en la escalera y, con la barbilla apoyada sobre la baranda,
dirigía una última mirada a aquella escena. Gregorio tomó impulso para
darle alcance, pero él debió de comprender su intención, pues, de un salto,
bajó varios escalones y desapareció, profiriendo unos alaridos que resonaron
por toda la escalera. Para colmo de males, la huida del jefe pareció trastornar
por completo al padre, que hasta entonces se había manteni


### 2. Preparación de las oraciones

Se normalizan saltos de línea y espacios que provienen del formato del PDF. La segmentación se hace después de punto, cierre de admiración o cierre de interrogación. Para no presentar como oraciones completas los cortes físicos del PDF, se eliminan el fragmento inicial que comienza en minúscula y el fragmento final que carece de puntuación de cierre.


In [3]:
texto_limpio = re.sub(r"\s+", " ", texto_paginas.replace("\f", " ")).strip()
candidatas = re.split(r"(?<=[.!?])\s+(?=[A-ZÁÉÍÓÚÜÑ¡¿—])", texto_limpio)

fragmentos_limite = []
oraciones = []
for candidata in candidatas:
    candidata = candidata.strip()
    es_inicial_incompleta = not re.match(r"^[A-ZÁÉÍÓÚÜÑ¡¿—]", candidata)
    tiene_cierre = bool(re.search(r'[.!?](?:[”’"])?$', candidata))
    if es_inicial_incompleta or not tiene_cierre:
        fragmentos_limite.append(candidata)
    else:
        oraciones.append(candidata)

print(f"Oraciones completas: {len(oraciones)}")
print(f"Fragmentos de límite excluidos: {len(fragmentos_limite)}")
display(pd.DataFrame({"id_oracion": range(1, len(oraciones) + 1), "oracion": oraciones}))

Oraciones completas: 50
Fragmentos de límite excluidos: 2


,id_oracion,oracion
0,1,—¡Madre!
1,2,¡Madre!
2,3,"—gimió Gregorio, mirándola desde abajo."
3,4,Por un momento se olvidó del gerente; y no pud...
4,5,"Su madre, gritando de nuevo y huyendo de la me..."
5,6,Pero Gregorio no podía dedicar ya su atención ...
6,7,"Gregorio tomó impulso para darle alcance, pero..."
7,8,"Para colmo de males, la huida del jefe pareció..."
8,9,"De nada le sirvieron a éste sus súplicas, que ..."
9,10,"La madre, a pesar del mal tiempo, había abiert..."


### 3. Carga del tokenizer y tokenización

BERT usa **WordPiece**. Antes de tokenizar, cada oración está formada por palabras lingüísticas y signos de puntuación. Después aparecen tokens especiales y, cuando una unidad no está en el vocabulario como una sola pieza, subpalabras marcadas normalmente con el prefijo ##.


In [4]:
NOMBRE_MODELO = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Tamaño del vocabulario: {tokenizer.vocab_size:,}")
print(f"Token inicial: {tokenizer.cls_token} (id={tokenizer.cls_token_id})")
print(f"Token separador: {tokenizer.sep_token} (id={tokenizer.sep_token_id})")

Tokenizer: BertTokenizer
Tamaño del vocabulario: 119,547
Token inicial: [CLS] (id=101)
Token separador: [SEP] (id=102)


In [5]:
filas = []
for i, oracion in enumerate(oraciones, start=1):
    palabras = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+|[^\w\s]", oracion, flags=re.UNICODE)
    tokens = tokenizer.tokenize(oracion, add_special_tokens=True)
    filas.append({
        "id_oracion": i,
        "palabras_y_signos": len(palabras),
        "tokens_bert": len(tokens),
        "oracion": oracion,
        "tokens": " ".join(tokens),
    })

tabla_tokens = pd.DataFrame(filas)
pd.set_option("display.max_colwidth", None)
display(tabla_tokens)

assert all(tokens.startswith("[CLS] ") and tokens.endswith(" [SEP]") for tokens in tabla_tokens["tokens"])
print("Verificación: todas las oraciones comienzan con [CLS] y terminan con [SEP].")

,id_oracion,palabras_y_signos,tokens_bert,oracion,tokens
0,1,4,6,—¡Madre!,[CLS] [UNK] ¡ Madre ! [SEP]
1,2,3,5,¡Madre!,[CLS] ¡ Madre ! [SEP]
2,3,8,14,"—gimió Gregorio, mirándola desde abajo.","[CLS] [UNK] gi ##mi ##ó Gregorio , mir ##ánd ##ola desde abajo . [SEP]"
3,4,29,45,"Por un momento se olvidó del gerente; y no pudo evita, ante el café vertido, abrir y cerrar repetidas veces las mandíbulas en el vacío.","[CLS] Por un momento se ol ##vid ##ó del ger ##ente ; y no pudo evit ##a , ante el café vert ##ido , abrir y ce ##rra ##r rep ##eti ##das veces las mand ##í ##bula ##s en el va ##cí ##o . [SEP]"
4,5,25,31,"Su madre, gritando de nuevo y huyendo de la mesa, se lanzó en brazos del padre, que corrió a su encuentro.","[CLS] Su madre , gr ##itan ##do de nuevo y huy ##endo de la mesa , se lanzó en brazos del padre , que cor ##rió a su encuentro . [SEP]"
5,6,36,47,"Pero Gregorio no podía dedicar ya su atención a sus padres; el gerente estaba en la escalera y, con la barbilla apoyada sobre la baranda, dirigía una última mirada a aquella escena.","[CLS] Pero Gregorio no podía dedicar ya su atención a sus padres ; el ger ##ente estaba en la es ##cale ##ra y , con la bar ##bil ##la apo ##yada sobre la bara ##nda , diri ##gía una última mira ##da a aquella escena . [SEP]"
6,7,37,53,"Gregorio tomó impulso para darle alcance, pero él debió de comprender su intención, pues, de un salto, bajó varios escalones y desapareció, profiriendo unos alaridos que resonaron por toda la escalera.","[CLS] Gregorio tomó impulso para dar ##le alcance , pero él debió de comprende ##r su intención , pues , de un salto , ba ##jó varios es ##cal ##ones y desa ##pare ##ció , prof ##iri ##endo unos ala ##ridos que reso ##naro ##n por toda la es ##cale ##ra . [SEP]"
7,8,118,153,"Para colmo de males, la huida del jefe pareció trastornar por completo al padre, que hasta entonces se había mantenido relativamente sereno; pues, en lugar de correr tras el fugitivo, o por lo menos permitir que así lo hiciese Gregorio, empuño con la diestra el bastón del gerente —que éste no había recogido, como tampoco su sombrero y su gabán, olvidados en una silla— y, armándose con la otra mano de un gran periódico que había sobre la mesa, se dispuso, dando fuertes patadas en el suelo, esgrimiendo papel y bastón, a hacer retroceder a Gregorio hasta el interior de su cuarto.","[CLS] Para col ##mo de males , la hui ##da del jefe pare ##ció tras ##tor ##nar por completo al padre , que hasta entonces se había man ##tenido relativamente ser ##eno ; pues , en lugar de correr tras el fu ##git ##ivo , o por lo menos permitir que así lo hi ##cies ##e Gregorio , em ##pu ##ño con la dies ##tra el bas ##tón del ger ##ente [UNK] que éste no había re ##co ##gido , como tampoco su som ##bre ##ro y su gab ##án , ol ##vidado ##s en una sil ##la [UNK] y , arm ##ándose con la otra mano de un gran periódico que había sobre la mesa , se dis ##puso , dando fuertes pat ##adas en el suelo , es ##gri ##mien ##do papel y bas ##tón , a hacer ret ##roc ##eder a Gregorio hasta el interior de su cuarto . [SEP]"
8,9,30,41,"De nada le sirvieron a éste sus súplicas, que no fueron entendidas; y aunque inclinó sumiso la cabeza, sólo consiguió excitar aún más a su padre.","[CLS] De nada le sir ##vieron a éste sus sú ##pli ##cas , que no fueron enten ##dida ##s ; y aunque in ##clin ##ó sum ##iso la cabeza , sólo consiguió ex ##citar aún más a su padre . [SEP]"
9,10,28,37,"La madre, a pesar del mal tiempo, había abierto una ventana y, violentamente inclinada hacia fuera, se cubría el rostro con las manos.","[CLS] La madre , a pesar del mal tiempo , había abierto una venta ##na y , violenta ##mente in ##clin ##ada hacia fuera , se cu ##br ##ía el ros ##tro con las manos . [SEP]"


Verificación: todas las oraciones comienzan con [CLS] y terminan con [SEP].


### 4. Palabras divididas en subpalabras

Para distinguir una palabra lingüística de sus tokens, se tokeniza cada palabra por separado. La tabla siguiente muestra únicamente las palabras que BERT divide en dos o más piezas. Por ejemplo, una palabra puede convertirse en una primera pieza y varias continuaciones ##....


In [6]:
divisiones = []
for id_oracion, oracion in enumerate(oraciones, start=1):
    palabras = re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+", oracion, flags=re.UNICODE)
    for palabra in palabras:
        piezas = tokenizer.tokenize(palabra, add_special_tokens=False)
        if len(piezas) > 1:
            divisiones.append({
                "id_oracion": id_oracion,
                "palabra_linguistica": palabra,
                "numero_piezas": len(piezas),
                "tokens_wordpiece": " ".join(piezas),
            })

tabla_subpalabras = pd.DataFrame(divisiones).drop_duplicates().reset_index(drop=True)
display(tabla_subpalabras)
print(f"Casos distintos de palabras divididas: {len(tabla_subpalabras)}")

,id_oracion,palabra_linguistica,numero_piezas,tokens_wordpiece
0,3,gimió,3,gi ##mi ##ó
1,3,mirándola,3,mir ##ánd ##ola
2,4,olvidó,3,ol ##vid ##ó
3,4,gerente,2,ger ##ente
4,4,evita,2,evit ##a
...,...,...,...,...
223,48,encendida,3,en ##cend ##ida
224,48,comedor,2,come ##dor
225,49,oía,2,o ##ía
226,50,oía,2,o ##ía


Casos distintos de palabras divididas: 228


### 5. Interpretación de la Parte A

- [CLS] se agrega al inicio de cada secuencia y [SEP] al final. No son palabras del texto: son tokens especiales que BERT necesita para estructurar la entrada.
- Una palabra lingüística no equivale necesariamente a un token. Las palabras frecuentes pueden ocupar una sola posición, mientras que palabras menos frecuentes, formas flexionadas o nombres pueden dividirse en varias piezas WordPiece.
- El prefijo ## indica que la pieza continúa la palabra anterior; no representa una palabra independiente.
- La puntuación también ocupa posiciones de token. Por ello, el número de tokens de BERT suele diferir del número de palabras lingüísticas.
- Esta diferencia será importante en la Parte B: las matrices de atención tendrán una fila y una columna por **token del modelo**, incluidos tokens especiales, puntuación y subpalabras, no una fila y columna por palabra lingüística.

**Alcance:** estos resultados describen únicamente la tokenización. Todavía no permiten concluir nada sobre capas, cabezas o patrones de atención.


---

## Parte B — Ejecución del modelo

En esta sección se carga el encoder BERT con output_attentions=True. El modelo se pone en modo evaluación y la inferencia se realiza dentro de torch.no_grad(), porque no se entrenarán sus parámetros.

Para asegurar que las matrices de atención estén disponibles se utiliza la implementación explícita eager. El corpus se procesa en lotes pequeños de cuatro oraciones para controlar el uso de memoria.

In [7]:
import math

import torch
from transformers import AutoModel
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
modelo = AutoModel.from_pretrained(
    NOMBRE_MODELO,
    output_attentions=True,
    attn_implementation="eager",
    local_files_only=True,
)
modelo.to(DISPOSITIVO)
modelo.eval()

print(f"PyTorch: {torch.__version__}")
print(f"Dispositivo: {DISPOSITIVO}")
print(f"Modelo: {type(modelo).__name__}")
print(f"Capas del encoder: {modelo.config.num_hidden_layers}")
print(f"Cabezas por capa: {modelo.config.num_attention_heads}")
print(f"Dimensión oculta: {modelo.config.hidden_size}")
print(f"Atenciones activadas: {modelo.config.output_attentions}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6684.39it/s]

PyTorch: 2.13.0+cpu
Dispositivo: cpu
Modelo: BertModel
Capas del encoder: 12
Cabezas por capa: 12
Dimensión oculta: 768
Atenciones activadas: True


### Ejecución sobre todas las oraciones

Las 50 oraciones completas obtenidas en la Parte A se pasan por el modelo. Dentro de cada lote, padding=True iguala temporalmente todas las secuencias a la longitud de la oración más larga de ese lote. La máscara de atención indica cuáles posiciones son texto real y cuáles son relleno.

Cada elemento de outputs.attentions corresponde a una capa y tiene forma:

**(tamaño del lote, número de cabezas, número de tokens, número de tokens)**.

Los dos últimos ejes forman una matriz cuadrada porque cada token de origen puede atender a cada token de destino.

In [8]:
TAMANO_LOTE = 4
resumen_lotes = []
atenciones_ejemplo = None
entradas_ejemplo = None

for inicio in range(0, len(oraciones), TAMANO_LOTE):
    oraciones_lote = oraciones[inicio:inicio + TAMANO_LOTE]
    entradas = tokenizer(
        oraciones_lote,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=tokenizer.model_max_length,
    )
    longitudes_reales = entradas["attention_mask"].sum(dim=1).tolist()
    entradas = {nombre: tensor.to(DISPOSITIVO) for nombre, tensor in entradas.items()}

    with torch.no_grad():
        outputs = modelo(**entradas)
        attentions = outputs.attentions

    assert attentions is not None
    assert len(attentions) == modelo.config.num_hidden_layers
    assert attentions[0].shape[1] == modelo.config.num_attention_heads
    assert attentions[0].shape[-1] == attentions[0].shape[-2]

    if atenciones_ejemplo is None:
        atenciones_ejemplo = tuple(atencion.cpu() for atencion in attentions)
        entradas_ejemplo = {nombre: tensor.cpu() for nombre, tensor in entradas.items()}

    resumen_lotes.append({
        "lote": len(resumen_lotes) + 1,
        "oraciones": f"{inicio + 1}-{inicio + len(oraciones_lote)}",
        "longitudes_reales": longitudes_reales,
        "numero_capas": len(attentions),
        "forma_por_capa": tuple(attentions[0].shape),
    })

tabla_lotes = pd.DataFrame(resumen_lotes)
display(tabla_lotes)
print(f"Total de oraciones procesadas: {len(oraciones)}")
print(f"Total de lotes: {len(tabla_lotes)}")

,lote,oraciones,longitudes_reales,numero_capas,forma_por_capa
0,1,1-4,"[6, 5, 14, 45]",12,"(4, 12, 45, 45)"
1,2,5-8,"[31, 47, 53, 153]",12,"(4, 12, 153, 153)"
2,3,9-12,"[41, 37, 56, 25]",12,"(4, 12, 56, 56)"
3,4,13-16,"[24, 12, 15, 35]",12,"(4, 12, 35, 35)"
4,5,17-20,"[35, 38, 39, 14]",12,"(4, 12, 39, 39)"
5,6,21-24,"[12, 34, 12, 18]",12,"(4, 12, 34, 34)"
6,7,25-28,"[27, 24, 33, 20]",12,"(4, 12, 33, 33)"
7,8,29-32,"[22, 14, 25, 15]",12,"(4, 12, 25, 25)"
8,9,33-36,"[16, 35, 35, 23]",12,"(4, 12, 35, 35)"
9,10,37-40,"[24, 57, 51, 44]",12,"(4, 12, 57, 57)"


Total de oraciones procesadas: 50
Total de lotes: 13


### Inspección detallada de las dimensiones

Se conserva el primer lote como evidencia concreta. Las atenciones son una tupla con una matriz por capa. En este BERT hay 12 matrices porque el encoder tiene 12 capas. Todas tienen 12 cabezas.

El número de tokens mostrado en la forma incluye [CLS], [SEP], puntuación, subpalabras y, al trabajar por lotes, posibles posiciones de relleno [PAD].

In [9]:
numero_capas = len(atenciones_ejemplo)
forma_primera_capa = tuple(atenciones_ejemplo[0].shape)
batch_size, numero_cabezas, numero_tokens, numero_tokens_destino = forma_primera_capa

print(f"Tipo de outputs.attentions: {type(atenciones_ejemplo).__name__}")
print(f"Número de matrices (capas): {numero_capas}")
print(f"Forma de la atención en la primera capa: {forma_primera_capa}")
print(f"  batch_size = {batch_size}")
print(f"  número_de_cabezas = {numero_cabezas}")
print(f"  número_de_tokens_origen = {numero_tokens}")
print(f"  número_de_tokens_destino = {numero_tokens_destino}")

formas_por_capa = pd.DataFrame({
    "capa": range(1, numero_capas + 1),
    "forma": [tuple(matriz.shape) for matriz in atenciones_ejemplo],
})
display(formas_por_capa)

assert numero_tokens == numero_tokens_destino
assert all(tuple(matriz.shape) == forma_primera_capa for matriz in atenciones_ejemplo)
print("Verificación: las matrices son cuadradas y todas las capas tienen la misma forma.")

Tipo de outputs.attentions: tuple
Número de matrices (capas): 12
Forma de la atención en la primera capa: (4, 12, 45, 45)
  batch_size = 4
  número_de_cabezas = 12
  número_de_tokens_origen = 45
  número_de_tokens_destino = 45


,capa,forma
0,1,"(4, 12, 45, 45)"
1,2,"(4, 12, 45, 45)"
2,3,"(4, 12, 45, 45)"
3,4,"(4, 12, 45, 45)"
4,5,"(4, 12, 45, 45)"
5,6,"(4, 12, 45, 45)"
6,7,"(4, 12, 45, 45)"
7,8,"(4, 12, 45, 45)"
8,9,"(4, 12, 45, 45)"
9,10,"(4, 12, 45, 45)"


Verificación: las matrices son cuadradas y todas las capas tienen la misma forma.


### Tokens y relleno del lote de ejemplo

La tabla permite relacionar las dimensiones con los tokens reales. La longitud válida se obtiene sumando la máscara de atención; cualquier posición adicional hasta la longitud máxima del lote es [PAD] y no pertenece a la oración original.

In [10]:
filas_ejemplo = []
for indice_lote in range(entradas_ejemplo["input_ids"].shape[0]):
    ids = entradas_ejemplo["input_ids"][indice_lote]
    longitud_valida = int(entradas_ejemplo["attention_mask"][indice_lote].sum())
    tokens_validos = tokenizer.convert_ids_to_tokens(ids[:longitud_valida].tolist())
    filas_ejemplo.append({
        "oracion": indice_lote + 1,
        "tokens_validos": longitud_valida,
        "posiciones_pad": int(ids.shape[0] - longitud_valida),
        "tokens": " ".join(tokens_validos),
    })

display(pd.DataFrame(filas_ejemplo))

,oracion,tokens_validos,posiciones_pad,tokens
0,1,6,39,[CLS] [UNK] ¡ Madre ! [SEP]
1,2,5,40,[CLS] ¡ Madre ! [SEP]
2,3,14,31,"[CLS] [UNK] gi ##mi ##ó Gregorio , mir ##ánd ##ola desde abajo . [SEP]"
3,4,45,0,"[CLS] Por un momento se ol ##vid ##ó del ger ##ente ; y no pudo evit ##a , ante el café vert ##ido , abrir y ce ##rra ##r rep ##eti ##das veces las mand ##í ##bula ##s en el va ##cí ##o . [SEP]"


### Conclusiones de la Parte B

- outputs.attentions contiene una tupla de **12 matrices**, una por cada capa del encoder.
- Cada matriz tiene cuatro dimensiones: lote, cabezas, tokens de origen y tokens de destino.
- El modelo posee **12 cabezas por capa**.
- Los dos ejes de tokens tienen el mismo tamaño, por lo que la atención de cada cabeza es una matriz cuadrada.
- La cantidad de tokens cambia entre lotes según la oración más larga y el relleno agregado.
- Las posiciones representan tokens de WordPiece, no necesariamente palabras lingüísticas completas.

En esta parte solamente se comprueba la ejecución y la estructura de las matrices. Identificar los tokens con mayor atención y comparar capas o cabezas corresponde a la Parte C.